# 04 — Exploration et comparaison de modèles de classement

**Projet** : Moteur de recommandation de catalogue e-commerce avec scikit-learn
**Modèle configuré** : `hist_gradient_boosting` (Gradient boosting par histogrammes (HistGradientBoostingClassifier))
**Pourquoi ce choix** : Le gradient boosting par histogrammes est le meilleur rapport précision/coût de scikit-learn pour scorer des couples (utilisateur, candidat) : il apprend sans transformation les **interactions** qui font la pertinence (affinité de catégorie x niveau de prix x intention récente), il est **natif valeurs manquantes** (un nouvel article sans historique de notation ne casse pas la publication), il traite les features catégorielles du catalogue sans one-hot massif — un encodage one-hot de 900 références exploserait la dimension et noierait le signal d'affinité — et il reste déterministe à graine fixée, ce qui est une exigence pour un moteur dont on audite les listes publiées. Ses limites sont documentées ici plutôt que cachées : il apprend un score **ponctuel** par couple et non une préférence relative, si bien qu'il n'optimise pas directement le NDCG (une perte de classement comme LambdaMART serait plus alignée) ; il ne construit aucune représentation latente partagée, donc il ne généralise pas à un utilisateur ou à un article jamais vu en entraînement autrement que par leurs attributs ; et il est sensible au déséquilibre de classes, la pertinence ne concernant qu'environ 14 % des candidats. La comparaison au notebook 04 avec une régression logistique (linéaire, lisible, mais aveugle aux interactions), une forêt aléatoire et un k-plus-proches voisins sur les embeddings de catégorie montre ce que l'on gagne et ce que l'on perd.

Un moteur de recommandation ne se choisit pas comme un classifieur. Il n'existe pas de
« meilleure exactitude » : il existe un **plancher** (le tirage aléatoire), une **référence
métier** (le tri par popularité, que le site sait déjà faire sans modèle), un **plafond**
(ce que la donnée permet de connaître au mieux) et une **coupure de publication** qui change la
question posée. Ce notebook installe ces quatre repères, puis compare les algorithmes à protocole
identique — mêmes candidats, mêmes utilisateurs, même coupure, mêmes graines.

## Objectifs pédagogiques

1. Mesurer le plancher, la référence métier et le plafond atteignable.
1. Comprendre pourquoi la métrique se calcule **par utilisateur**, jamais globalement.
1. Comparer les algorithmes de la stack à protocole identique, dispersion comprise.
1. Choisir la coupure K en connaissance de cause (qualité contre diversité).
1. Vérifier l'absence de fuite par une sonde univariée.

**Objectifs transverses du dépôt**

- Construire un jeu de candidats (utilisateur x article) plutôt qu'une matrice d'interactions, et comprendre ce que ce choix autorise : consommer les features de fiche article, scorer un article neuf, publier sous contraintes métier.
- Formaliser le contrat d'antériorité de chaque feature : connue avant la session (fiche article, historique agrégé), connue au moment de la session (intention récente), ou interdite (issue de la session elle-même).
- Comprendre pourquoi l'unité d'évaluation est l'utilisateur et non la ligne : une précision calculée sur toutes les lignes mêle un client très actif à un nouveau venu et ne décrit aucun des deux.

## 0. Mise en place

Même configuration que `python -m src.main`, à une différence près : le volume est réduit
(`NB_ROWS`) pour que le notebook s'exécute en quelques secondes. Les distributions restent
réalistes, mais les écarts fins ne sont plus lisibles — c'est assumé, et la section 7 mesure
précisément ce que le bruit autorise à conclure.

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous
# les lignes INFO de production. Les erreurs réelles restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (12000 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 12000

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.45)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    # Le pré-traitement supprime la colonne de groupe (identifiant, non modélisable), or une
    # métrique de classement se calcule **par groupe** puis se moyenne. Elle doit donc voyager à
    # côté des matrices, exactement comme dans `TrainPipeline` et dans les fixtures de tests.
    # `None` pour toute tâche sans structure de groupe : le comportement des autres projets est
    # inchangé.
    group_column = getattr(config.data, "group_column", None)

    def groups_of(split: pd.DataFrame | None) -> Any:
        if split is None or not group_column or group_column not in split.columns:
            return None
        return split[group_column].to_numpy()

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "groups_train": groups_of(enriched["train"]),
        "groups_val": groups_of(enriched["val"]),
        "groups_test": groups_of(enriched["test"]),
        "feature_names": list(pipeline.feature_names_out),
        # Colonnes de la matrice **avant** pré-traitement (donc avant one-hot). Indispensables dès
        # qu'un notebook ré-applique le pipeline à un nouveau cadre : sélectionner les colonnes de
        # `X_train` (après one-hot) sur un cadre enrichi lève un KeyError sur les modalités.
        "frame_columns": list(X_train_frame.columns),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

## 1. Le protocole : des métriques par utilisateur

Une liste de recommandation se juge **liste par liste**. Les fonctions ci-dessous sont écrites
dans le notebook, et non importées, pour que leur définition soit lisible : le NDCG normalise le
gain par le gain idéal, la précision et le rappel se lisent à la coupure K, et la moyenne se fait
sur les utilisateurs — pas sur les lignes.

In [ ]:
import pandas as pd


def discounts(size: int) -> np.ndarray:
    """Return the NDCG positional discounts 1 / log2(rank + 1).

    Le discount logarithmique est ce qui distingue le NDCG d'une simple précision : être pertinent
    en première position vaut plus qu'en dixième, parce que l'attention de l'utilisateur décroît
    avec la position. log2(rank + 1) est la forme standard, retenue parce qu'elle décroît vite au
    début de la liste et lentement ensuite — exactement le profil d'attention observé.
    """
    if size <= 0:
        return np.zeros(0)
    return 1.0 / np.log2(np.arange(2, size + 2, dtype="float64"))


def dcg(gains: np.ndarray) -> float:
    """Return the discounted cumulative gain of an ordered gain vector."""
    gains = np.asarray(gains, dtype="float64")
    return float((gains * discounts(gains.size)).sum()) if gains.size else 0.0


def ndcg_at_k(truth: np.ndarray, score: np.ndarray, k: int) -> float:
    """Return the NDCG@K of one list: realised gain over the best possible gain.

    Retourner 0 quand la liste ne contient rien de pertinent est un choix : un utilisateur sans
    intention ne peut pas être satisfait, et compter sa liste comme « parfaite » récompenserait un
    moteur qui publie peu. Ces utilisateurs sont donc exclus du calcul par l'appelant.
    """
    truth = np.asarray(truth, dtype="float64")
    order = np.argsort(-np.asarray(score, dtype="float64"), kind="stable")
    ideal = dcg(np.sort(truth)[::-1][:k])
    return dcg(truth[order][:k]) / ideal if ideal > 0 else 0.0


def precision_at_k(truth: np.ndarray, score: np.ndarray, k: int) -> float:
    """Return the share of the K published slots that are relevant."""
    order = np.argsort(-np.asarray(score, dtype="float64"), kind="stable")
    selected = np.asarray(truth, dtype="float64")[order][:k]
    return float(selected.sum() / selected.size) if selected.size else 0.0


def recall_at_k(truth: np.ndarray, score: np.ndarray, k: int) -> float:
    """Return the share of the available relevance captured by the K published slots."""
    truth = np.asarray(truth, dtype="float64")
    total = float(truth.sum())
    if total <= 0:
        return float("nan")
    order = np.argsort(-np.asarray(score, dtype="float64"), kind="stable")
    return float(truth[order][:k].sum() / total)


def average_precision_at_k(truth: np.ndarray, score: np.ndarray, k: int) -> float:
    """Return AP@K: the mean of the precisions measured at each relevant hit."""
    order = np.argsort(-np.asarray(score, dtype="float64"), kind="stable")
    selected = np.asarray(truth, dtype="float64")[order][:k]
    if selected.sum() <= 0:
        return 0.0
    precisions = np.cumsum(selected) / np.arange(1, selected.size + 1, dtype="float64")
    return float((precisions * selected).sum() / min(selected.sum(), k))


def hit_rate_at_k(truth: np.ndarray, score: np.ndarray, k: int) -> float:
    """Return 1 when at least one relevant item lands in the K published slots."""
    order = np.argsort(-np.asarray(score, dtype="float64"), kind="stable")
    return 1.0 if float(np.asarray(truth, dtype="float64")[order][:k].sum()) > 0 else 0.0


def ranking_metrics(frame: pd.DataFrame, score_column: str, k: int = 10) -> pd.Series:
    """Compute every ranking metric **per user**, then average.

    C'est la fonction la plus importante du notebook : elle matérialise le fait que l'unité
    d'évaluation est l'utilisateur. Les utilisateurs sans aucun candidat pertinent sont exclus,
    parce qu'aucun classement ne peut les satisfaire.
    """
    rows = {"ndcg": [], "precision": [], "recall": [], "map": [], "hit": []}
    for _, group in frame.groupby("user_id", observed=True, sort=False):
        truth = group[TARGET].to_numpy(dtype="float64")
        if truth.sum() <= 0:
            continue
        score = group[score_column].to_numpy(dtype="float64")
        rows["ndcg"].append(ndcg_at_k(truth, score, k))
        rows["precision"].append(precision_at_k(truth, score, k))
        rows["recall"].append(recall_at_k(truth, score, k))
        rows["map"].append(average_precision_at_k(truth, score, k))
        rows["hit"].append(hit_rate_at_k(truth, score, k))
    if not rows["ndcg"]:
        return pd.Series(dtype="float64")
    values = pd.Series({key: float(np.nanmean(item)) for key, item in rows.items()})
    values["utilisateurs"] = float(len(rows["ndcg"]))
    return values.round(4)


def score_frame(PREPARED: dict[str, Any], split: str, model: Any) -> pd.DataFrame:
    """Return the enriched split rows with the model score, ready for per-user metrics.

    Le cadre enrichi (avant pré-traitement) porte les identifiants et les colonnes métier ; la
    matrice pré-traitée porte ce que le modèle consomme. Les deux sont alignés ligne à ligne, ce
    qui permet de scorer sur l'une et d'analyser sur l'autre.
    """
    frame = PREPARED["enriched"][split].reset_index(drop=True)
    matrix = PREPARED[f"X_{split}"]
    probabilities = model.predict_proba(matrix)
    scored = frame.copy()
    scored["score"] = probabilities[:, -1] if probabilities.ndim == 2 else probabilities
    return scored

In [ ]:
from src.models import build_model

MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
# `build_model(params=...)` **remplace** les réglages de `conf/model/default.yaml`. Pour ne faire
# varier qu'un seul facteur à la fois dans les sections suivantes (algorithme, coupure, graine,
# grille), on part donc toujours de cette copie des réglages configurés.
CONFIGURED_PARAMS = dict(CONFIG.model.params)
FIT_RESULT = MODEL.fit(
    PREPARED["X_train"],
    PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    groups=PREPARED.get("groups_train"),
    groups_val=PREPARED.get("groups_val"),
    callbacks=[],
)
print(MODEL.summary())
print(f"entraînement : {FIT_RESULT.duration_seconds:.2f} s")

Le score de classement n'a pas besoin d'être une probabilité : il lui faut être
ordonnable. La fonction ci-dessous prend ce que l'estimateur sait produire, ce qui permet de
comparer loyalement un arbre (probabilité) et un SVM (distance à la frontière). Les colonnes
utilisées ensuite sont résolues depuis la configuration, jamais écrites dans le notebook.

In [ ]:
# Aucun nom de colonne n'est écrit dans ce notebook : l'évaluateur du projet résout ses réglages
# depuis `conf/config.yaml` (bloc `recommendation`, avec repli sur le noeud `data`). Le notebook
# utilise exactement les mêmes colonnes que le rapport de production — changer le schéma ne
# demande qu'un changement de configuration, pas une édition de notebook.
from src.evaluation.evaluator import RankingSettings

CONFIG_DICT = CONFIG.model_dump(mode="json")
SETTINGS = RankingSettings.resolve(CONFIG_DICT)


def rank_scores(model: Any, matrix: pd.DataFrame) -> np.ndarray:
    """Return a continuous ranking score, whatever the estimator exposes.

    Un score de classement n'a pas besoin d'être une probabilité : il lui faut seulement être
    **ordonnable**. Un SVM sans calibration probabiliste classe très bien par sa distance à la
    frontière, un arbre par sa probabilité, un modèle de factorisation par son produit scalaire.
    Cette fonction prend ce que l'estimateur sait produire, dans l'ordre de préférence.
    """
    try:
        probabilities = np.asarray(model.predict_proba(matrix), dtype="float64")
        return probabilities[:, -1].ravel() if probabilities.ndim == 2 else probabilities.ravel()
    except Exception:
        estimator = getattr(model, "estimator_", None) or getattr(model, "model_", None)
        for attribute in ("decision_function", "score_samples", "predict"):
            function = getattr(estimator, attribute, None)
            if callable(function):
                values = np.asarray(function(matrix), dtype="float64")
                return values[:, -1].ravel() if values.ndim == 2 else values.ravel()
        raise


VAL = PREPARED["enriched"]["val"].reset_index(drop=True).copy()
VAL["score"] = rank_scores(MODEL, PREPARED["X_val"])
TARGET = CONFIG.data.target
GROUPE = SETTINGS.group_column
ARTICLE = SETTINGS.item_column
TOP_K = int(SETTINGS.top_k)
print(f"split de validation : {len(VAL):,} candidats, {VAL[GROUPE].nunique()} utilisateurs")
print(f"coupure publiée     : top-{TOP_K}")
print(
    f"colonnes résolues   : groupe={GROUPE} article={ARTICLE} "
    f"popularité={SETTINGS.popularity_column} intention={SETTINGS.intent_column}"
)

## 2. Plancher, référence métier et modèle

Quatre moteurs, un seul protocole. Le tirage aléatoire borne ce qu'on peut obtenir sans
information ; le tri par popularité est ce que le site sait déjà faire sans modèle ; l'intention
récente est l'exploitation pure (remontrer ce que la personne a vu). **Un modèle qui ne bat pas
la popularité de façon nette ne justifie pas son coût de mise en œuvre.**

In [ ]:
# Quatre moteurs, un seul protocole : mêmes candidats, mêmes utilisateurs, même coupure.
rng = np.random.default_rng(CONFIG.seed)
VAL["score_aleatoire"] = rng.random(len(VAL))
VAL["score_popularite"] = VAL[SETTINGS.popularity_column].astype("float64")
VAL["score_intention"] = VAL[SETTINGS.intent_column].astype("float64")

references = {
    "tirage aléatoire": "score_aleatoire",
    "tri par popularité": "score_popularite",
    "intention récente": "score_intention",
    f"modèle configuré ({MODEL.algorithm})": "score",
}
table = pd.DataFrame(
    {name: ranking_metrics(VAL, column, TOP_K) for name, column in references.items()}
).T
table["couverture catalogue"] = [
    round(
        VAL.assign(
            rang=VAL.groupby(GROUPE, observed=True)[column].rank(ascending=False, method="first")
        )
        .query("rang <= @TOP_K")[ARTICLE]
        .nunique()
        / VAL[ARTICLE].nunique(),
        4,
    )
    for column in references.values()
]
display(table)

plancher = float(table.loc["tirage aléatoire", "ndcg"])
reference = float(table.loc["tri par popularité", "ndcg"])
modele = float(table.loc[f"modèle configuré ({MODEL.algorithm})", "ndcg"])
print(f"plancher (aléatoire)      : {plancher:.4f}")
print(f"référence (popularité)    : {reference:.4f}")
print(f"modèle                    : {modele:.4f}")
print(f"gain sur la popularité    : {(modele - reference) / reference:+.1%}")

**Ce qu'il faut retenir**

- L'aléatoire n'est pas zéro : avec des candidats de volumes comparables, il obtient un NDCG non nul. Toute progression se mesure à partir de ce plancher.
- La référence à battre est la popularité, pas l'aléatoire. C'est elle qui figure au contrat du projet.
- Un moteur qui se contente de redisposer les articles les plus vus n'ajoute rien : l'écart à la référence est la seule justification du modèle.

## 3. Le plafond : ce que la donnée permet de connaître

Le générateur synthétique connaît la probabilité de pertinence **avant** bruit : il peut donc
classer parfaitement tout ce qui est connaissable. Ce classement oracle définit le plafond, et la
distance qui reste n'est pas une marge de progression — c'est du bruit irréductible. Sur données
réelles ce plafond n'est pas publié : on ne compare alors qu'aux références, et c'est précisément
pourquoi la section 2 existe.

In [ ]:
# Le générateur connaît la probabilité de pertinence **avant** bruit : il peut donc classer
# parfaitement tout ce qui est connaissable. Ce classement oracle définit le plafond atteignable,
# et la distance qui reste n'est pas une marge de progression — c'est du bruit irréductible.
import json

# Le plafond n'est publié que par le générateur synthétique : sur données réelles il n'existe pas.
# La variable est donc initialisée à NaN **avant** la branche conditionnelle, pour que les cellules
# suivantes puissent la tester sans dépendre de l'existence du fichier.
plafond = float("nan")

metadata_path = PATHS.data_file("generation_metadata", fmt="json", stage="raw")
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    plafond = float(metadata.get("ndcg_ceiling_oracle") or float("nan"))
    publie = {
        "aléatoire": metadata.get("ndcg_baseline_random"),
        "popularité": metadata.get("ndcg_baseline_popularity"),
        "plafond oracle": plafond,
        "marge totale": metadata.get("ndcg_headroom"),
        "coupure de référence": metadata.get("top_k_reference"),
    }
    display(pd.Series(publie, name=f"NDCG@{TOP_K} publié par le générateur").to_frame())

    chemin = plafond - plancher
    progression = (modele - plancher) / chemin if chemin > 0 else float("nan")
    print(f"le modèle a parcouru {progression:.1%} du chemin entre l'aléatoire et le plafond")
    print(f"part du plafond capturée : {modele / plafond:.1%}")
else:
    print("Aucune métadonnée de génération : le plafond n'est pas publié.")
    print("Sur données réelles c'est le cas normal — on ne compare alors qu'aux références.")

**Ce qu'il faut retenir**

- Un modèle qui capture une large part du plafond est au bout de ce que la donnée autorise : la suite du travail est sur les features, pas sur les hyperparamètres.
- Le plafond sépare « le modèle est faible » de « la donnée est bruitée ». Sans lui, on optimise du bruit.

## 4. Lecture globale contre lecture par utilisateur

C'est la démonstration la plus importante du notebook. La métrique « globale » n'est pas une
version bruitée de la métrique par utilisateur : c'est **une autre question**, qui pondère chaque
individu par son volume de candidats.

In [ ]:
# La même donnée, deux lectures. C'est la démonstration la plus importante du notebook : la
# métrique « globale » n'est pas une version bruitée de la métrique par utilisateur, c'est une
# AUTRE question, qui mélange des utilisateurs de volumes très différents.
par_utilisateur = ranking_metrics(VAL, "score", TOP_K)

utilisateurs = VAL[GROUPE].nunique()
ordre = np.argsort(-VAL["score"].to_numpy(), kind="stable")
verite = VAL[TARGET].to_numpy(dtype="float64")
global_precision = float(verite[ordre][: TOP_K * utilisateurs].mean())
global_ndcg = ndcg_at_k(verite, VAL["score"].to_numpy(), TOP_K * utilisateurs)

volumes = VAL.groupby(GROUPE, observed=True).size()
print("lecture GLOBALE (toutes lignes confondues)")
print(f"  précision sur les {TOP_K}x{volumes.size} premiers rangs : {global_precision:.4f}")
print(f"  NDCG sur une liste unique de {TOP_K * volumes.size} places  : {global_ndcg:.4f}")
print()
print(f"lecture PAR UTILISATEUR (moyenne de {int(par_utilisateur['utilisateurs'])} listes)")
print(f"  précision@{TOP_K} : {par_utilisateur['precision']:.4f}")
print(f"  NDCG@{TOP_K}      : {par_utilisateur['ndcg']:.4f}")
print()
print(f"Écart de précision : {par_utilisateur['precision'] - global_precision:+.4f}")
print()
print("Pourquoi la lecture par utilisateur est la seule valide ici :")
display(
    pd.DataFrame(
        {
            "utilisateurs": volumes.value_counts().sort_index(),
        }
    )
    .rename_axis("candidats par utilisateur")
    .reset_index()
)
print("Un utilisateur très actif contribue à des dizaines de lignes, un nouveau venu à une")
print("seule : la moyenne globale est donc pondérée par l'activité, et décrit le comportement")
print("du moteur sur les gros clients — pas le service rendu à un utilisateur moyen.")

**Ce qu'il faut retenir**

- Une précision globale mélange des utilisateurs très actifs et des nouveaux venus : elle décrit le moteur sur les gros clients, pas le service rendu à un utilisateur moyen.
- Toute métrique de classement se moyenne sur l'unité de décision — ici l'utilisateur, ailleurs la session ou la requête.
- C'est aussi ce qui rend l'AUC inadaptée : elle compare des paires d'articles appartenant à des utilisateurs différents, paires que personne ne verra jamais côte à côte.

## 5. Sensibilité à la coupure K

Publier 5 ou 20 articles ne mesure pas la même qualité. Cette table est l'outil de décision du K :
elle montre le compromis structurel (la précision chute, le rappel monte) et son prix en diversité
(la couverture progresse, la concentration recule).

In [ ]:
# Publier 5 ou 20 articles ne mesure pas la même qualité. Cette table est l'outil de décision
# du K : elle montre le compromis structurel (la précision chute, le rappel monte) et le prix en
# diversité (la couverture catalogue progresse, la concentration recule).
lignes = []
for coupure in (3, 5, TOP_K, 15, 20):
    valeurs = ranking_metrics(VAL, "score", coupure)
    published = VAL.assign(
        rang=VAL.groupby(GROUPE, observed=True)["score"].rank(ascending=False, method="first")
    ).query("rang <= @coupure")
    counts = published[ARTICLE].value_counts()
    shares = counts.to_numpy(dtype="float64") / float(counts.sum())
    lignes.append(
        {
            "K": coupure,
            "NDCG": valeurs["ndcg"],
            "précision": valeurs["precision"],
            "rappel": valeurs["recall"],
            "MAP": valeurs["map"],
            "hit-rate": valeurs["hit"],
            "couverture catalogue": round(counts.size / VAL[ARTICLE].nunique(), 4),
            "Herfindahl": round(float((shares**2).sum()), 4),
            "retenu": coupure == TOP_K,
        }
    )
coupures = pd.DataFrame(lignes).set_index("K")
display(coupures)
print(f"K retenu par la configuration : {TOP_K}")

**Ce qu'il faut retenir**

- Le NDCG varie peu avec K quand le haut de liste est bon : c'est la signature d'un classement correct, pas d'un hasard heureux.
- La précision chute mécaniquement avec K, le rappel monte : choisir K sur la seule précision revient à choisir K = 1, donc à ne rien recommander.
- Le K retenu est une contrainte produit (surface d'affichage réelle), déclarée dans `conf/config.yaml` — pas un hyperparamètre à optimiser.

## 6. Comparaison des algorithmes à protocole identique

`params={{}}` construit chaque algorithme avec ses réglages par défaut : les `model.params`
configurés sont propres au boosting par histogrammes et les injecter ailleurs lèverait une erreur.
Le modèle configuré **et réglé** est mesuré en section 2 ; ici on compare les familles.

In [ ]:
import warnings

from src.models.factory import available_algorithms

ALGORITHMS = available_algorithms(CONFIG.metrics.task)
print(f"{len(ALGORITHMS)} algorithmes servent la tâche '{CONFIG.metrics.task}' :")
print(ALGORITHMS)

lignes = []
avertissements: dict[str, dict[str, int]] = {}
# Comparaison **loyale** : `params={}` construit chaque algorithme avec ses réglages par défaut.
# Les `model.params` configurés sont propres au boosting par histogrammes (`max_leaf_nodes`,
# `max_features`, `learning_rate`) : les injecter dans une régression logistique lèverait une
# erreur, et les régler à la main pour chaque concurrent fausserait le classement. Le modèle
# configuré ET réglé est, lui, mesuré en section 1.
for algorithm in ALGORITHMS:
    try:
        # Les avertissements des bibliothèques sont capturés puis restitués en fin de cellule :
        # les masquer cacherait une information diagnostique (convergence, classes rares), les
        # laisser inonder la sortie noierait le tableau. Ni l'un ni l'autre.
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always")
            candidate = build_model(
                CONFIG, feature_names=PREPARED["feature_names"], algorithm=algorithm, params={}
            )
            result = candidate.fit(
                PREPARED["X_train"],
                PREPARED["y_train"],
                X_val=PREPARED["X_val"],
                y_val=PREPARED["y_val"],
                groups=PREPARED.get("groups_train"),
                groups_val=PREPARED.get("groups_val"),
                callbacks=[],
            )
            scores = rank_scores(candidate, PREPARED["X_val"])
        comptes: dict[str, int] = {}
        for item in captured:
            premiere = str(item.message).strip().splitlines()[0][:78]
            cle = f"{item.category.__name__} : {premiere}"
            comptes[cle] = comptes.get(cle, 0) + 1
        if comptes:
            avertissements[algorithm] = comptes
        mesure = ranking_metrics(VAL.assign(score_candidat=scores), "score_candidat", TOP_K)
        published = (
            VAL.assign(
                rang=pd.Series(scores, index=VAL.index)
                .groupby(VAL[GROUPE])
                .rank(ascending=False, method="first")
            )
            .query("rang <= @TOP_K")[ARTICLE]
            .nunique()
        )
        lignes.append(
            {
                "algorithme": algorithm,
                f"NDCG@{TOP_K}": mesure["ndcg"],
                f"précision@{TOP_K}": mesure["precision"],
                f"rappel@{TOP_K}": mesure["recall"],
                "couverture": round(published / VAL[ARTICLE].nunique(), 4),
                "secondes": round(result.duration_seconds, 2),
            }
        )
    except Exception as error:  # un algorithme incompatible ne doit pas casser l'exploration
        lignes.append(
            {
                "algorithme": algorithm,
                f"NDCG@{TOP_K}": float("nan"),
                f"précision@{TOP_K}": float("nan"),
                f"rappel@{TOP_K}": float("nan"),
                "couverture": float("nan"),
                "secondes": float("nan"),
            }
        )
        avertissements[algorithm] = {f"échec : {type(error).__name__} : {error}"[:120]: 1}

comparaison = (
    pd.DataFrame(lignes).set_index("algorithme").sort_values(f"NDCG@{TOP_K}", ascending=False)
)
display(comparaison)
print(f"référence à battre (popularité) : {reference:.4f}")
print(f"plafond atteignable             : {plafond:.4f}" if not np.isnan(plafond) else "")

if avertissements:
    print("\n--- avertissements et échecs capturés ---")
    for algorithm, comptes in avertissements.items():
        for message, nombre in comptes.items():
            print(f"  {algorithm:<24} x{nombre}  {message}")

**Ce qu'il faut retenir**

- Un algorithme qui échoue est conservé dans le tableau avec ses avertissements : masquer un échec, c'est perdre une information diagnostique.
- Le classement se lit avec la dispersion de la section 7 : un écart inférieur à l'écart-type entre graines n'est pas un gain.
- La couverture accompagne chaque ligne : un algorithme plus précis mais plus concentré coûte en diversité, et ce coût est réel.

## 7. Stabilité entre graines

Trois graines suffisent, sur le volume d'un notebook, à estimer la dispersion d'un même
algorithme. Cette dispersion est l'étalon de toute comparaison : en dessous, on mesure un tirage.

In [ ]:
# Un écart entre deux algorithmes inférieur à la dispersion entre graines n'est pas un gain :
# c'est un tirage. Trois graines suffisent à estimer cette dispersion sur un volume de notebook.
lignes = []
for graine in (CONFIG.seed, CONFIG.seed + 1, CONFIG.seed + 2):
    candidate = build_model(
        CONFIG,
        feature_names=PREPARED["feature_names"],
        params={**CONFIGURED_PARAMS, "random_state": graine},
    )
    candidate.fit(
        PREPARED["X_train"],
        PREPARED["y_train"],
        X_val=PREPARED["X_val"],
        y_val=PREPARED["y_val"],
        groups=PREPARED.get("groups_train"),
        groups_val=PREPARED.get("groups_val"),
        callbacks=[],
    )
    scores = rank_scores(candidate, PREPARED["X_val"])
    mesure = ranking_metrics(VAL.assign(score_graine=scores), "score_graine", TOP_K)
    lignes.append(
        {
            "graine": graine,
            f"NDCG@{TOP_K}": mesure["ndcg"],
            f"précision@{TOP_K}": mesure["precision"],
            f"rappel@{TOP_K}": mesure["recall"],
        }
    )
graines = pd.DataFrame(lignes).set_index("graine")
display(graines)

ecart_type = float(graines[f"NDCG@{TOP_K}"].std(ddof=1))
print(
    f"NDCG moyen {graines[f'NDCG@{TOP_K}'].mean():.4f} | écart-type entre graines {ecart_type:.4f}"
)
print(f"amplitude  {graines[f'NDCG@{TOP_K}'].max() - graines[f'NDCG@{TOP_K}'].min():.4f}")
print()
print("Tout écart entre deux algorithmes inférieur à cette dispersion ne justifie pas un")
print("changement de modèle : il justifie un travail sur les features.")

**Ce qu'il faut retenir**

- Choisir le vainqueur d'une grille sur une seule graine, c'est choisir un bruit : la décision doit survivre au changement de graine.
- La dispersion entre graines est aussi le plancher de ce qu'on peut promettre en production : un gain annoncé plus petit ne sera pas vérifiable.

## 8. Sonde de fuite

Aucune colonne observable ne doit approcher le plafond. Si une seule y parvenait, elle porterait
l'information que le générateur garde pour lui — et le modèle « appris » ne serait qu'un proxy de
la réponse. Le test est univarié volontairement : c'est le plus sévère.

In [ ]:
# Sonde de fuite : aucune colonne observable ne doit approcher le plafond. Si une seule y
# parvenait, elle porterait l'information que le générateur garde pour lui — et le modèle
# « appris » ne serait qu'un proxy de la réponse.
colonnes = [
    column
    for column in VAL.columns
    if column not in {TARGET, "score", "score_aleatoire", "score_popularite", "score_intention"}
    and pd.api.types.is_numeric_dtype(VAL[column])
]
lignes = []
for column in colonnes:
    valeurs = VAL[column].astype("float64")
    if valeurs.nunique() < 2:
        continue
    mesure = ranking_metrics(VAL.assign(sonde=valeurs), "sonde", TOP_K)
    lignes.append({"colonne": column, f"NDCG@{TOP_K}": mesure["ndcg"]})
sondes = (
    pd.DataFrame(lignes).set_index("colonne").sort_values(f"NDCG@{TOP_K}", ascending=False).head(12)
)
display(sondes)

meilleure = float(sondes[f"NDCG@{TOP_K}"].iloc[0])
print(f"meilleure colonne seule  : {meilleure:.4f}")
print(f"référence popularité     : {reference:.4f}")
print(f"modèle complet           : {modele:.4f}")
print(f"plafond oracle           : {plafond:.4f}" if not np.isnan(plafond) else "")
print()
if not np.isnan(plafond):
    print(
        f"la meilleure colonne atteint {meilleure / plafond:.1%} du plafond : "
        "aucune fuite univariée."
    )
    print("Le modèle complet dépasse nettement chaque colonne prise isolément : il combine")
    print("l'information, il ne la recopie pas.")

**Ce qu'il faut retenir**

- La meilleure colonne seule reste proche de la référence de popularité : c'est attendu, l'audience est le signal observable le plus fort.
- Le modèle complet dépasse nettement chaque colonne isolée : il combine l'information, il ne la recopie pas.
- Sur données réelles, ce test se refait à chaque nouvelle feature — une fuite s'introduit par un joint de données, pas par un algorithme.

## 9. Grille d'hyperparamètres

La grille est déclarée dans le manifeste du projet et centrée sur le point retenu, pour que le
notebook **rejoue** la décision au lieu de la paraphraser.

In [ ]:
# Grille explorée par ce notebook : elle est **déclarée dans le manifeste du projet**
# (`extras.notebook_param_grid`), donc versionnée avec lui, centrée sur le point retenu. Le
# notebook rejoue ainsi la décision d'hyperparamètres au lieu de la paraphraser.
GRID: dict[str, list[Any]] = {
    "max_leaf_nodes": [15, 31],
    "max_features": [0.7, 1.0],
    "learning_rate": [0.04, 0.08],
}

if not GRID:
    print("Aucune grille déclarée pour ce projet : la recherche d'hyperparamètres n'est pas")
    print("rejouée dans le notebook, le point configuré est celui de conf/model/default.yaml.")
else:
    import itertools

    from src.models import build_model

    combinaisons = list(itertools.product(*[GRID[key] for key in GRID]))
    print(f"{len(combinaisons)} combinaisons explorées : {GRID}")

    lignes = []
    for combinaison in combinaisons:
        params = {**CONFIGURED_PARAMS, **dict(zip(GRID.keys(), combinaison, strict=True))}
        candidate = build_model(CONFIG, feature_names=PREPARED["feature_names"], params=params)
        candidate.fit(
            PREPARED["X_train"],
            PREPARED["y_train"],
            X_val=PREPARED["X_val"],
            y_val=PREPARED["y_val"],
            groups=PREPARED.get("groups_train"),
            groups_val=PREPARED.get("groups_val"),
            callbacks=[],
        )
        scores = rank_scores(candidate, PREPARED["X_val"])
        mesure = ranking_metrics(VAL.assign(score_grille=scores), "score_grille", TOP_K)
        lignes.append(
            {
                **params,
                f"NDCG@{TOP_K}": mesure["ndcg"],
                f"précision@{TOP_K}": mesure["precision"],
                f"rappel@{TOP_K}": mesure["recall"],
            }
        )

    colonnes_grille = list(GRID.keys())
    grille = pd.DataFrame(lignes)
    display(
        grille[[*colonnes_grille, f"NDCG@{TOP_K}", f"précision@{TOP_K}", f"rappel@{TOP_K}"]]
        .sort_values(f"NDCG@{TOP_K}", ascending=False)
        .reset_index(drop=True)
    )
    meilleur = grille.loc[grille[f"NDCG@{TOP_K}"].idxmax(), colonnes_grille].to_dict()
    configure = {key: CONFIGURED_PARAMS.get(key) for key in colonnes_grille}
    print(f"meilleur point de la grille : {meilleur}")
    print(f"point configuré             : {configure}")
    ecart = float(grille[f"NDCG@{TOP_K}"].max() - grille[f"NDCG@{TOP_K}"].min())
    print(f"amplitude NDCG sur toute la grille : {ecart:.4f}")

    # La dispersion entre graines mesurée en section 7 borne ce qu'on peut conclure. Elle est
    # lue depuis l'espace global pour que la cellule reste exécutable hors ordre.
    bruit = globals().get("ecart_type")
    if meilleur == configure:
        print("Le point configuré est bien celui que la grille préfère : la décision rejoue.")
    elif bruit is not None and ecart <= float(bruit):
        print(f"AUCUN point de la grille ne se détache : l'amplitude totale ({ecart:.4f}) reste")
        print(f"sous la dispersion entre graines ({float(bruit):.4f}). Le point configuré est donc")
        print("défendable, et chercher un meilleur réglage sur ce volume serait ajuster du bruit.")
    else:
        print("Le point configuré n'est pas le meilleur de cette grille, et l'écart dépasse la")
        print("dispersion entre graines : à trancher en connaissance de cause (coût, variance,")
        print("couverture), pas sur le seul NDCG.")

**Ce qu'il faut retenir**

- Le point configuré doit sortir de la grille : si ce n'est pas le cas, la décision est à documenter (coût, variance, couverture).
- La grille du notebook est volontairement étroite : la recherche large appartient au pipeline, pas à un notebook pédagogique.

## Conclusion

Quatre repères ont été installés : le plancher aléatoire, la référence de popularité, le plafond
oracle et la coupure de publication. Les algorithmes ont été comparés à protocole identique, la
dispersion entre graines a borné ce qu'on peut conclure, et la sonde de fuite a écarté le scénario
le plus dangereux — un modèle qui recopie la réponse.

Le notebook 05 entraîne le modèle retenu sur le volume complet, avec les callbacks et la
journalisation de production. Le notebook 06 analyse les erreurs de classement et formule les
recommandations.